In [1]:
import torch
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = 204
model = timm.create_model("vit_small_patch16_224", pretrained=True, num_classes=num_classes).to(device)
checkpoint = torch.load("D:\\Tesi\\FirstFineTuning\\best_model.pth")
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [2]:
from torchvision.datasets import ImageFolder
from torchvision import transforms

data_config = timm.data.resolve_model_data_config(model)
imagenet_mean, imagenet_std = data_config["mean"], data_config["std"]

search_transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(mean=imagenet_mean, std=imagenet_std)])

path = "D:\\Tesi\\Sets\\Set1\\search"
batch_size = 128

search_set = ImageFolder(root=path, transform=search_transform)
search_loader = torch.utils.data.DataLoader(search_set, batch_size=batch_size, shuffle=False, num_workers=1)

In [3]:
print(model)

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [4]:
from NAS.NAS_Utils import set_initial_masks, count_parameters, compute_obj
from torch.nn import CrossEntropyLoss

set_initial_masks(model)
print(count_parameters(model))
print(compute_obj(model, CrossEntropyLoss(), device, search_loader))

21.744203567504883
-4.573562165774069


from PruneUtils import importance_score

weights = [
    model.patch_embed.proj.weight[2],
    model.patch_embed.proj.bias[2],
    model.patch_embed.proj.weight[3],
    model.patch_embed.proj.bias[3],
]

grads = [
    model.patch_embed.proj.weight.grad[2],
    model.patch_embed.proj.bias.grad[2],
    model.patch_embed.proj.weight.grad[3],
    model.patch_embed.proj.bias.grad[3]
]
print(importance_score(weights, grads))

In [5]:
from PruneUtils import head_alignment, HeadAligned, WeightBias

head_aligned = head_alignment(model.blocks[0].attn)
print(head_aligned.Q.bias.shape)


torch.Size([6, 64])
